# Task 1 Model Comparison

**Purpose.** Consolidate saved held-out results without retraining or test-driven selection.

This is a self-contained coursework notebook. It does not import custom helper modules and does not depend on another notebook. It defaults to a small `smoke` run; switch `RUN_MODE` to `final` only after the smoke run succeeds.

Results and takeaways must be written from executed outputs. No performance claims are pre-filled.

## Goal

Combine already-executed model outputs without retraining or changing any test result.

## Setup

Run this notebook last. It reads the stable metrics contract written by each model notebook.

In [ ]:
# Install only missing packages. Neural-network notebooks do not install or use
# TensorFlow, PyTorch, Keras, or scikit-learn neural-network implementations.
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
}
missing = [pip_name for module, pip_name in required_packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Environment ready.")

In [ ]:
import copy
import hashlib
import json
import math
import os
import random
import time
import warnings
import zipfile
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import resample_poly

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

IN_COLAB = (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    default_root = Path("/content/drive/MyDrive/ProgMathsAI")
else:
    default_root = Path.cwd()
    if not (default_root / "Fake Music Caps.zip").exists() and (default_root.parent / "Fake Music Caps.zip").exists():
        default_root = default_root.parent

# These environment variables provide a local-test override without changing
# the Colab defaults shown to a coursework reader.
PROJECT_ROOT = Path(os.environ.get("PROGMATHSAI_PROJECT_ROOT", str(default_root)))
ZIP_PATH = Path(os.environ.get("FAKEMUSICCAPS_ZIP_PATH", str(PROJECT_ROOT / "Fake Music Caps.zip")))
DATA_DIR = Path(os.environ.get("FAKEMUSICCAPS_DATA_DIR", str(PROJECT_ROOT / "Fake Music Caps")))
OUTPUT_ROOT = Path(os.environ.get("FAKEMUSICCAPS_OUTPUT_ROOT", str(PROJECT_ROOT / "task1_outputs")))

RUN_MODE = os.environ.get("FAKEMUSICCAPS_RUN_MODE", "smoke").lower()
if RUN_MODE not in {"smoke", "final"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'final'.")

MODEL_TAG = "comparison"
OUTPUT_DIR = OUTPUT_ROOT / MODEL_TAG
FIGURES_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_DIR / "cache"
for directory in (FIGURES_DIR, METRICS_DIR, MODELS_DIR, TABLES_DIR, CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

FOLDER_TO_LABEL = {
    "audioldm2": 0,
    "MusicGen_medium": 1,
    "musicldm": 2,
    "mustango": 3,
    "stable_audio_open": 4,
}
CLASS_NAMES = ["AudioLDM2", "MusicGen", "MusicLDM", "Mustango", "Stable Audio Open"]

SAMPLE_RATE = 16_000
CLIP_SECONDS = 2.5
FRAME_MS = 25
HOP_MS = 10
N_FFT = 512
N_MELS = 26
N_MFCC = 20
POOL_FACTOR = 4
SMOKE_PER_CLASS_PER_SPLIT = 8

print({
    "run_mode": RUN_MODE,
    "project_root": str(PROJECT_ROOT),
    "zip_path": str(ZIP_PATH),
    "data_dir": str(DATA_DIR),
    "output_dir": str(OUTPUT_DIR),
})

## Results

Only existing result files are included; missing model runs are reported explicitly.

In [ ]:
expected_models = {
    "logistic_regression": "Logistic Regression",
    "svm": "Support Vector Machine",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "numpy_mlp": "NumPy MLP",
    "numpy_cnn": "NumPy CNN",
    "numpy_lstm": "NumPy LSTM",
}

rows, missing = [], []
for folder, display_name in expected_models.items():
    path = OUTPUT_ROOT / folder / "metrics" / f"test_metrics_{RUN_MODE}.json"
    if not path.exists():
        missing.append(display_name)
        continue
    payload = json.loads(path.read_text(encoding="utf-8"))
    test = payload["test"]
    rows.append({
        "model": payload["model"],
        "best_variant": payload["best_variant"],
        "accuracy": test["accuracy"],
        "balanced_accuracy": test["balanced_accuracy"],
        "macro_precision": test["macro_precision"],
        "macro_recall": test["macro_recall"],
        "macro_f1": test["macro_f1"],
        "runtime_seconds": payload["runtime_seconds"],
        "parameter_count": payload["parameter_count"],
    })

if missing:
    print("Missing results:", ", ".join(missing))
if not rows:
    raise FileNotFoundError("No model results were found. Run model notebooks first.")

comparison = pd.DataFrame(rows).sort_values(
    ["macro_f1", "balanced_accuracy"], ascending=False).reset_index(drop=True)
display(comparison)
comparison.to_csv(TABLES_DIR / f"model_comparison_{RUN_MODE}.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5))
positions = np.arange(len(comparison))
width = 0.38
ax.bar(positions - width / 2, comparison["macro_f1"], width, label="Macro F1")
ax.bar(positions + width / 2, comparison["balanced_accuracy"], width, label="Balanced accuracy")
ax.set_xticks(positions, comparison["model"], rotation=35, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Held-out test score")
ax.set_title(f"Task 1 model comparison ({RUN_MODE} run)")
ax.legend()
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"model_comparison_{RUN_MODE}.png", dpi=160)
plt.show()

## Takeaways

Use the `final` comparison—not smoke-mode scores—in the report. Discuss accuracy together with macro-F1, runtime, model complexity, and generator-level confusion.